# Problem 2: Regression (California House Prices)
**Objective:** Predict the continuous `Median House Value` variable using multiple features.
**Steps:**
1. Fetch and process the data.
2. Split data: 70% Train, 15% Validation, 15% Test.
3. Scale the features (important for regularization models like Ridge/Lasso).
4. Apply standard Linear Regression, Ridge Regression, and Lasso Regression.
5. Report Mean Squared Error (MSE) and Mean Absolute Error (MAE).

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.metrics import mean_absolute_error , mean_squared_error
from sklearn.preprocessing import StandardScaler


In [4]:
DataSet = pd.read_csv("California_Houses.csv")
DataSet.head()
DataSet.info()
DataSet.shape
DataSet.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Median_House_Value        20640 non-null  float64
 1   Median_Income             20640 non-null  float64
 2   Median_Age                20640 non-null  int64  
 3   Tot_Rooms                 20640 non-null  int64  
 4   Tot_Bedrooms              20640 non-null  int64  
 5   Population                20640 non-null  int64  
 6   Households                20640 non-null  int64  
 7   Latitude                  20640 non-null  float64
 8   Longitude                 20640 non-null  float64
 9   Distance_to_coast         20640 non-null  float64
 10  Distance_to_LA            20640 non-null  float64
 11  Distance_to_SanDiego      20640 non-null  float64
 12  Distance_to_SanJose       20640 non-null  float64
 13  Distance_to_SanFrancisco  20640 non-null  float64
dtypes: flo

,Median_House_Value,Median_Income,Median_Age,Tot_Rooms,Tot_Bedrooms,Population,Households,Latitude,Longitude,Distance_to_coast,Distance_to_LA,Distance_to_SanDiego,Distance_to_SanJose,Distance_to_SanFrancisco
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,2.064000e+04,2.064000e+04,20640.000000,20640.000000
mean,206855.816909,3.870671,28.639486,2635.763081,537.898014,1425.476744,499.539680,35.631861,-119.569704,40509.264883,2.694220e+05,3.981649e+05,349187.551219,386688.422291
std,115395.615874,1.899822,12.585558,2181.615252,421.247906,1132.462122,382.329753,2.135952,2.003532,49140.039160,2.477324e+05,2.894006e+05,217149.875026,250122.192316
min,14999.000000,0.499900,1.000000,2.000000,1.000000,3.000000,1.000000,32.540000,-124.350000,120.676447,4.205891e+02,4.849180e+02,569.448118,456.141313
25%,119600.000000,2.563400,18.000000,1447.750000,295.000000,787.000000,280.000000,33.930000,-121.800000,9079.756762,3.211125e+04,1.594264e+05,113119.928682,117395.477505
50%,179700.000000,3.534800,29.000000,2127.000000,435.000000,1166.000000,409.000000,34.260000,-118.490000,20522.019101,1.736675e+05,2.147398e+05,459758.877000,526546.661701
75%,264725.000000,4.743250,37.000000,3148.000000,647.000000,1725.000000,605.000000,37.710000,-118.010000,49830.414479,5.271562e+05,7.057954e+05,516946.490963,584552.007907
max,500001.000000,15.000100,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,41.950000,-114.310000,333804.686371,1.018260e+06,1.196919e+06,836762.678210,903627.663298


In [9]:
X = DataSet.drop("Median_House_Value",axis=1)
y = DataSet["Median_House_Value"]

X


,Median_Income,Median_Age,Tot_Rooms,Tot_Bedrooms,Population,Households,Latitude,Longitude,Distance_to_coast,Distance_to_LA,Distance_to_SanDiego,Distance_to_SanJose,Distance_to_SanFrancisco
0,8.3252,41,880,129,322,126,37.88,-122.23,9263.040773,556529.158342,735501.806984,67432.517001,21250.213767
1,8.3014,21,7099,1106,2401,1138,37.86,-122.22,10225.733072,554279.850069,733236.884360,65049.908574,20880.600400
2,7.2574,52,1467,190,496,177,37.85,-122.24,8259.085109,554610.717069,733525.682937,64867.289833,18811.487450
3,5.6431,52,1274,235,558,219,37.85,-122.25,7768.086571,555194.266086,734095.290744,65287.138412,18031.047568
4,3.8462,52,1627,280,565,259,37.85,-122.25,7768.086571,555194.266086,734095.290744,65287.138412,18031.047568
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,1.5603,25,1665,374,845,330,39.48,-121.09,162031.481121,654530.186299,830631.543047,248510.058162,222619.890417
20636,2.5568,18,697,150,356,114,39.49,-121.21,160445.433537,659747.068444,836245.915229,246849.888948,218314.424634
20637,1.7000,17,2254,485,1007,433,39.43,-121.22,153754.341182,654042.214020,830699.573163,240172.220489,212097.936232
20638,1.8672,18,1860,409,741,349,39.43,-121.32,152005.022239,657698.007703,834672.461887,238193.865909,207923.199166


In [24]:
# 2. Split dataset (70% Train, 15% Validation, 15% Test)
# First split: 70% Train, 30% Temporary
X_train , X_temp , y_train , y_temp = train_test_split(X , y , test_size=0.3 , random_state=0)
X_val , X_test , y_val , y_test = train_test_split(X_temp , y_temp , test_size=0.5 , random_state=0)


In [25]:
# 3. Standardize the data 
# Ridge and Lasso apply penalties to feature coefficients; scaling ensures fair penalization.
ScalerReg = StandardScaler()
X_train_scaled = ScalerReg.fit_transform(X_train)
X_val_scaled = ScalerReg.transform(X_val)
X_test_scaled = ScalerReg.transform(X_test)


In [26]:
# %%
# 4 & 5. Tune Models using Validation Set, then Evaluate on Test Set

print("--- Tuning Ridge Regression on Validation Set ---")
alphas_to_test = [0.01, 0.1, 1.0, 10.0, 100.0]

best_ridge_alpha = None
best_ridge_mse = float('inf')

for a in alphas_to_test:
    # Train on TRAINING set
    temp_ridge = Ridge(alpha=a)
    temp_ridge.fit(X_train_scaled, y_train)
    
    # Evaluate on VALIDATION set
    val_preds = temp_ridge.predict(X_val_scaled)
    val_mse = mean_squared_error(y_val, val_preds)
    
    if val_mse < best_ridge_mse:
        best_ridge_mse = val_mse
        best_ridge_alpha = a

print(f"Best Ridge alpha found: {best_ridge_alpha} (Validation MSE: {best_ridge_mse:.4f})")


print("\n--- Tuning Lasso Regression on Validation Set ---")
best_lasso_alpha = None
best_lasso_mse = float('inf')

for a in alphas_to_test:
    # Train on TRAINING set
    temp_lasso = Lasso(alpha=a, max_iter=10000)
    temp_lasso.fit(X_train_scaled, y_train)
    
    # Evaluate on VALIDATION set
    val_preds = temp_lasso.predict(X_val_scaled)
    val_mse = mean_squared_error(y_val, val_preds)
    
    if val_mse < best_lasso_mse:
        best_lasso_mse = val_mse
        best_lasso_alpha = a

print(f"Best Lasso alpha found: {best_lasso_alpha} (Validation MSE: {best_lasso_mse:.4f})\n")

--- Tuning Ridge Regression on Validation Set ---
Best Ridge alpha found: 1.0 (Validation MSE: 4780951996.8544)

--- Tuning Lasso Regression on Validation Set ---
Best Lasso alpha found: 10.0 (Validation MSE: 4780669552.4166)



In [27]:
# 4. Initialize Models
# Note: For Ridge and Lasso, the alpha parameter controls the penalty. We will use a default alpha=1.0 for comparison.
models = {
    "Linear Regression": LinearRegression(), # Linear regression doesn't have an alpha to tune
    "Ridge Regression": Ridge(alpha=best_ridge_alpha),
    "Lasso Regression": Lasso(alpha=best_lasso_alpha, max_iter=10000)
}

In [28]:
results = {}
print("--- Regression Models Evaluation (Test Set) ---")
for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Predict on the test set
    preds = model.predict(X_test_scaled)
    
    # Calculate Metrics
    mse = mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    results[name] = {"MSE": mse, "MAE": mae}
    
    print(f"{name}:")
    print(f"  MSE: {mse:.4f}")
    print(f"  MAE: {mae:.4f}\n")

--- Regression Models Evaluation (Test Set) ---
Linear Regression:
  MSE: 4832043335.6653
  MAE: 49751.7625

Ridge Regression:
  MSE: 4831771014.9974
  MAE: 49749.8122

Lasso Regression:
  MSE: 4831639758.0618
  MAE: 49746.2696

